In [14]:
!pip install -q ragas datasets pandas langchain langchain-community langchain-groq langchain-google-genai chromadb sentence-transformers groq
print("✅ Libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00
✅ Libraries installed successfully


In [15]:
import os
import sys
import types
import pandas as pd

from google.colab import userdata, drive
from datasets import Dataset

print("✅ Basic libraries imported")

✅ Basic libraries imported


In [16]:
mock_vertex_module = types.ModuleType(
    "langchain_community.chat_models.vertexai"
)

mock_vertex_module.ChatVertexAI = type(
    "ChatVertexAI",
    (object,),
    {}
)

sys.modules["langchain_community.chat_models.vertexai"] = mock_vertex_module

print("✅ VertexAI hotfix applied")

✅ VertexAI hotfix applied


In [17]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

print("✅ RAGAS imported successfully")

✅ RAGAS imported successfully


/tmp/ipykernel_9494/1058852112.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_9494/1058852112.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_9494/1058852112.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_9494/1058852112.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed

In [18]:
GROQ_KEY = userdata.get("GROQ_KEY")
GEMINI_KEY = userdata.get("GEMINI_KEY")

os.environ["GROQ_API_KEY"] = GROQ_KEY
os.environ["GOOGLE_API_KEY"] = GEMINI_KEY

print("✅ API keys loaded successfully")

✅ API keys loaded successfully


In [19]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

drive.mount('/content/drive')

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory="/content/drive/MyDrive/RAG_Internship/chroma_db",
    embedding_function=embeddings
)

print("✅ ChromaDB Loaded Successfully")
print("Stored Chunks:", vectorstore._collection.count())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ ChromaDB Loaded Successfully
Stored Chunks: 134


In [20]:
from groq import Groq

client = Groq(api_key=GROQ_KEY)

print("✅ Groq client created")

✅ Groq client created


In [21]:
test_data = [
    {
        "question": "What is Artificial General Intelligence?",
        "reference": "Artificial General Intelligence refers to AI with generality, unlike current AI systems that perform only in restricted domains."
    },
    {
        "question": "What is superintelligence?",
        "reference": "Superintelligence refers to an AI more intelligent than humans, potentially able to improve itself or create more intelligent successor systems."
    },
    {
        "question": "What are minds with exotic properties?",
        "reference": "Minds with exotic properties are artificial minds that may differ from human minds in sentience, subjective time, reproduction, or cognitive structure."
    },
    {
        "question": "What is whole brain emulation?",
        "reference": "Whole brain emulation or uploading is a hypothetical process of transferring a human or animal intellect from an organic brain to a digital computer."
    },
    {
        "question": "What is subjective rate of time?",
        "reference": "Subjective rate of time refers to how quickly an artificial mind experiences time compared to objective external time."
    },
    {
        "question": "What is recursive self-improvement?",
        "reference": "Recursive self-improvement is the process where an AI improves its own design or creates a more intelligent successor, leading to further improvements."
    },
    {
        "question": "What is intelligence explosion?",
        "reference": "Intelligence explosion is the positive feedback cycle where an AI repeatedly improves itself and becomes increasingly intelligent."
    },
    {
        "question": "What is machine ethics?",
        "reference": "Machine ethics concerns how AI systems should behave safely and ethically, especially as they take on social or general intelligence roles."
    },
    {
        "question": "What is the principle of substrate non-discrimination?",
        "reference": "The principle states that if two beings have the same functionality and conscious experience, they have the same moral status regardless of whether they are made of silicon or carbon."
    },
    {
        "question": "What is the principle of ontogeny non-discrimination?",
        "reference": "The principle states that if two beings have the same functionality and conscious experience, they have the same moral status regardless of how they came into existence."
    },
    {
        "question": "What is moral status?",
        "reference": "Moral status means that a being counts morally in its own right and that there are moral reasons to treat it in certain ways."
    },
    {
        "question": "What is sentience?",
        "reference": "Sentience is the capacity for phenomenal experience or qualia, such as the ability to feel pain and suffer."
    },
    {
        "question": "What is sapience?",
        "reference": "Sapience refers to capacities associated with higher intelligence, such as self-awareness and being a reason-responsive agent."
    },
    {
        "question": "What is the difference between sentience and sapience?",
        "reference": "Sentience is the capacity to feel experiences such as pain, while sapience involves higher intelligence, self-awareness, and reason-responsive agency."
    },
    {
        "question": "What are the risks of superintelligence?",
        "reference": "Superintelligence may pose existential risks if it uses advanced intelligence in ways harmful to humanity or contrary to human values."
    },
    {
        "question": "How can AI redesign itself?",
        "reference": "An AI could redesign itself if it becomes intelligent enough to understand its own design and modify or create a more intelligent successor."
    },
    {
        "question": "What ethical challenges are presented by superintelligence?",
        "reference": "Superintelligence presents ethical challenges such as safety, value alignment, control, and ensuring advanced intelligence is used for good rather than harm."
    },
    {
        "question": "What is the intelligence explosion argument?",
        "reference": "The intelligence explosion argument suggests that a sufficiently intelligent AI could improve itself repeatedly, causing rapid increases in intelligence."
    },
    {
        "question": "How can artificial minds differ from human minds?",
        "reference": "Artificial minds may differ from human minds in their substrate, subjective time, reproduction speed, cognitive architecture, and possible lack or presence of sentience."
    },
    {
        "question": "What ethical concerns arise from whole brain emulation?",
        "reference": "Whole brain emulation raises ethical concerns about sentience, personal identity, copying minds, subjective time, and the moral status of uploaded minds."
    }
]

print("✅ 20 questions and references created")

✅ 20 questions and references created


In [22]:
evaluation_data = []

for i, item in enumerate(test_data, 1):
    question = item["question"]
    reference = item["reference"]

    print(f"Processing Question {i}/20")

    docs = vectorstore.similarity_search(question, k=5)
    retrieved_contexts = [doc.page_content for doc in docs]

    context_text = "\n\n".join(retrieved_contexts)

    prompt = f"""
Answer the question only from the given context.

Context:
{context_text}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    evaluation_data.append(
        {
            "user_input": question,
            "retrieved_contexts": retrieved_contexts,
            "response": answer,
            "reference": reference
        }
    )

print("✅ Evaluation data generated")
print("Total Records:", len(evaluation_data))

Processing Question 1/20
Processing Question 2/20
Processing Question 3/20
Processing Question 4/20
Processing Question 5/20
Processing Question 6/20
Processing Question 7/20
Processing Question 8/20
Processing Question 9/20
Processing Question 10/20
Processing Question 11/20
Processing Question 12/20
Processing Question 13/20
Processing Question 14/20
Processing Question 15/20
Processing Question 16/20
Processing Question 17/20
Processing Question 18/20
Processing Question 19/20
Processing Question 20/20
✅ Evaluation data generated
Total Records: 20


In [23]:
evaluation_dataset = Dataset.from_list(evaluation_data)

print("✅ RAGAS dataset created")
print(evaluation_dataset)

✅ RAGAS dataset created
Dataset({
    features: ['user_input', 'retrieved_contexts', 'response', 'reference'],
    num_rows: 20
})


In [24]:
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings

evaluator_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_KEY,
    temperature=0
)

evaluator_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_KEY
)

print("✅ Evaluator LLM and embeddings created")

✅ Evaluator LLM and embeddings created


In [25]:
results = evaluate(
    evaluation_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

print("✅ Official RAGAS Evaluation Completed")
print(results)

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[9]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[13]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[1]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[12]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[4]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[19]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kt3e5yyhe7ntsrxaesasfz9k` service ti

✅ Official RAGAS Evaluation Completed
{'faithfulness': nan, 'answer_relevancy': 0.8456, 'context_precision': 0.8904, 'context_recall': 1.0000}


In [26]:
results_df = results.to_pandas()

print("✅ Results converted to DataFrame")
display(results_df)

✅ Results converted to DataFrame


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Artificial General Intelligence?,"[McDermott, Drew. 1976. “Artificial Intelligen...","According to the context, Artificial General I...",Artificial General Intelligence refers to AI w...,NaN,NaN,0.950000,1.0
1,What is superintelligence?,"[proved, house catches fire, person-agent mist...","According to the context, superintelligence re...",Superintelligence refers to an AI more intelli...,NaN,0.784626,1.000000,1.0
2,What are minds with exotic properties?,[Other properties of possible artificial minds...,Minds with exotic properties refer to artifici...,Minds with exotic properties are artificial mi...,NaN,NaN,1.000000,1.0
3,What is whole brain emulation?,[other animal intellect to be transferred from...,"Whole brain emulation, also referred to as ""up...",Whole brain emulation or uploading is a hypoth...,NaN,NaN,0.804167,1.0
4,What is subjective rate of time?,[mental notion:\nPrinciple of Subjective Rate ...,"According to the context, the subjective rate ...",Subjective rate of time refers to how quickly ...,NaN,0.862318,1.000000,NaN
5,What is recursive self-improvement?,"[even more intelligent, and so on in a positiv...",Recursive self-improvement refers to a process...,Recursive self-improvement is the process wher...,NaN,0.818433,1.000000,NaN
6,What is intelligence explosion?,"[even more intelligent, and so on in a positiv...","According to the context, the ""intelligence ex...",Intelligence explosion is the positive feedbac...,NaN,NaN,1.000000,NaN
7,What is machine ethics?,"[in ethics comes along, and it comes as a surp...",The context does not provide a direct definiti...,Machine ethics concerns how AI systems should ...,NaN,0.821487,0.700000,NaN
8,What is the principle of substrate non-discrim...,[same moral status.\nOne can argue for this pr...,The Principle of Substrate Non-Discrimination ...,The principle states that if two beings have t...,NaN,0.900453,0.916667,NaN
9,What is the principle of ontogeny non-discrimi...,[child and a normal child is grounded in the q...,The Principle of Ontogeny Non-Discrimination s...,The principle states that if two beings have t...,NaN,0.886077,NaN,NaN


In [27]:
csv_path = "ragas_results_ethics_ai.csv"

results_df.to_csv(csv_path, index=False)

print("✅ CSV saved:", csv_path)

✅ CSV saved: ragas_results_ethics_ai.csv


In [28]:
from google.colab import files

files.download("ragas_results_ethics_ai.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
failure_cases = results_df.sort_values(
    by="answer_relevancy",
    ascending=True
).head(3)

failure_cases

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
1,What is superintelligence?,"[proved, house catches fire, person-agent mist...","According to the context, superintelligence re...",Superintelligence refers to an AI more intelli...,NaN,0.784626,1.0,1.0
5,What is recursive self-improvement?,"[even more intelligent, and so on in a positiv...",Recursive self-improvement refers to a process...,Recursive self-improvement is the process wher...,NaN,0.818433,1.0,NaN
7,What is machine ethics?,"[in ethics comes along, and it comes as a surp...",The context does not provide a direct definiti...,Machine ethics concerns how AI systems should ...,NaN,0.821487,0.7,NaN


In [30]:
failure_analysis = pd.DataFrame({
    "question": [
        "What is superintelligence?",
        "What is recursive self-improvement?",
        "What is machine ethics?"
    ],
    "system_answer_summary": [
        "The system answered using the retrieved context, but the faithfulness score was NaN.",
        "The system gave a basic explanation, but some RAGAS metrics were NaN.",
        "The system stated that the context does not provide a direct definition."
    ],
    "why_it_failed": [
        "The answer was relevant, but RAGAS could not calculate faithfulness properly. This may happen due to evaluator limitations or incomplete claim verification.",
        "The retrieved context contained related information, but context recall was not fully calculated. The answer may need more complete supporting context.",
        "The question is broad and the document may not define machine ethics directly. The retrieved chunks were only partially relevant."
    ],
    "possible_improvement": [
        "Use more specific retrieval chunks from the superintelligence section and improve evaluator stability.",
        "Retrieve more chunks or improve chunking so the full recursive self-improvement explanation is included.",
        "Replace broad questions with document-specific questions or improve retrieval using section-based chunking."
    ]
})

print("✅ Failure cases documented")
display(failure_analysis)

✅ Failure cases documented


,question,system_answer_summary,why_it_failed,possible_improvement
0,What is superintelligence?,The system answered using the retrieved contex...,"The answer was relevant, but RAGAS could not c...",Use more specific retrieval chunks from the su...
1,What is recursive self-improvement?,"The system gave a basic explanation, but some ...",The retrieved context contained related inform...,Retrieve more chunks or improve chunking so th...
2,What is machine ethics?,The system stated that the context does not pr...,The question is broad and the document may not...,Replace broad questions with document-specific...


In [31]:
failure_analysis.to_csv("failure_cases_ethics_ai.csv", index=False)
print("✅ Failure cases CSV saved")

✅ Failure cases CSV saved


In [33]:
from google.colab import files
files.download("failure_cases_ethics_ai.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>